## 🎯 Learning Objectives
* Design and implement a multi-agent research system using LangGraph.
* Utilize a supervisor agent for dynamic routing and task orchestration between specialized sub-agents.
* Define and manage complex agent states for iterative information flow.
* Integrate mock tools to simulate external functionalities within an agentic workflow.
* Construct and debug advanced LangGraph topologies for production-grade agent systems.


## Exercise: Build a Multi-Agent Research System with Supervisor Routing

### Task Description

Your goal is to construct a sophisticated multi-agent research system using LangGraph. This system will take a research query as input, dynamically route it through a series of specialized agents, and produce a comprehensive research report. The core of this system will be a `Supervisor` agent responsible for orchestrating the workflow, deciding which sub-agent should act next, and determining when the research task is complete.

### System Requirements

1.  **LangGraph Foundation**: The entire system must be built using LangGraph for state management and agent orchestration.
2.  **Supervisor Agent**: Implement a central `Supervisor` agent. This agent will use an LLM to analyze the current state of the research and decide the next action. Its decisions will include:
    *   Routing to a specialized sub-agent (e.g., `Researcher`, `Analyzer`, `ReportWriter`).
    *   Instructing an agent to re-evaluate or perform more work (e.g., `Researcher` needs to do more search).
    *   Terminating the research process when a satisfactory report is generated.
3.  **Specialized Sub-Agents**: Implement at least three distinct sub-agents:
    *   **`Researcher`**: Simulates web searches to gather raw information relevant to the query. It should update the shared state with its findings.
    *   **`Analyzer`**: Processes the raw information gathered by the `Researcher`, synthesizes key insights, and identifies gaps. It should update the shared state with its analysis.
    *   **`ReportWriter`**: Takes the synthesized analysis and generates a structured, coherent research report. It should update the shared state with the final report.
4.  **Iterative Workflow**: The system must support iterative refinement. For example, the `Analyzer` might determine that more research is needed, routing control back to the `Supervisor`, which then sends it back to the `Researcher`.
5.  **Mock Tools**: Simulate external tools (e.g., web search API, data analysis library) using simple Python functions. Do not integrate actual external APIs for this exercise.
6.  **Well-Defined State**: Design a clear and comprehensive `AgentState` that effectively passes information between agents.

### Evaluation Criteria

*   **Correctness**: Does the LangGraph correctly implement the specified routing logic and agent interactions?
*   **Robustness**: Can the system handle iterative tasks and produce a coherent output for various queries?
*   **Modularity**: Are the agents well-defined with clear responsibilities, promoting reusability and maintainability?
*   **Supervisor Effectiveness**: Does the `Supervisor` agent make intelligent routing decisions based on the current state?
*   **Code Quality**: Is the code clean, well-commented, and adheres to best practices?
*   **Output Quality**: Is the final research report generated by the `ReportWriter` comprehensive and relevant to the initial query?

Good luck!


In [ ]:
import os
import json
from typing import TypedDict, List, Annotated, Union
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.chat_models import ChatOpenAI # Using a common LLM, replace with your preferred provider in 2026
from langgraph.graph import StateGraph, END

# --- Configuration and Mock Setup ---

# Set your API key for the LLM provider. In a real 2026 scenario, this would be managed securely.
# For this exercise, we'll use a mock LLM or a placeholder for demonstration.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Mock LLM for deterministic behavior in an exercise
class MockChatOpenAI:
    def __init__(self, model_name="gpt-4o-2026-preview", temperature=0.0):
        self.model_name = model_name
        self.temperature = temperature

    def invoke(self, prompt):
        # Simulate LLM response based on prompt content
        if "supervisor" in prompt.lower() and "decide the next agent" in prompt.lower():
            # Simple routing logic for mock supervisor
            if "research_notes" not in prompt and "query" in prompt:
                return AIMessage(content='{"next_agent": "researcher"}')
            elif "research_notes" in prompt and "analysis_results" not in prompt:
                return AIMessage(content='{"next_agent": "analyzer"}')
            elif "analysis_results" in prompt and "final_report" not in prompt:
                return AIMessage(content='{"next_agent": "report_writer"}')
            else:
                return AIMessage(content='{"next_agent": "FINISH"}')
        elif "researcher" in prompt.lower() and "gather information" in prompt.lower():
            query = prompt.split("Query: ")[1].split("\n")[0]
            return AIMessage(content=f"Simulated research notes for '{query}': Key findings include A, B, and C. Some sources suggest D. Further investigation into E might be beneficial.")
        elif "analyzer" in prompt.lower() and "synthesize" in prompt.lower():
            notes = prompt.split("Research Notes: ")[1].split("\n")[0]
            return AIMessage(content=f"Analysis of notes '{notes[:50]}...': The core theme is X, supported by A and B. C is a conflicting point. A deeper dive into E is recommended to resolve ambiguities.")
        elif "report writer" in prompt.lower() and "generate a report" in prompt.lower():
            analysis = prompt.split("Analysis Results: ")[1].split("\n")[0]
            return AIMessage(content=f"## Research Report\n\n### Executive Summary\nBased on the analysis of '{analysis[:50]}...', this report details the findings related to the initial query. Key insights include X, A, and B. Further research on E is suggested.\n\n### Detailed Findings\n[Detailed findings based on A, B, C, D, E]\n\n### Conclusion\n[Summary of conclusions]\n")
        return AIMessage(content="Mock LLM response.")

# Uncomment the line below and set your API key if you want to use a real LLM
# llm = ChatOpenAI(model="gpt-4o-2026-preview", temperature=0.0)
llm = MockChatOpenAI() # Using the mock LLM for this exercise

# --- Define Agent State ---

class AgentState(TypedDict):
    """Represents the state of our multi-agent research system."""
    query: str  # The initial research query
    research_notes: str # Accumulated raw research findings
    analysis_results: str # Synthesized analysis and insights
    final_report: str # The final generated research report
    next_agent: str # The name of the next agent to route to
    iterations: int # Counter for tracking iterations to prevent infinite loops
    messages: Annotated[List[BaseMessage], RunnablePassthrough] # For conversational history (optional, but good practice)

# --- Mock Tool Functions (Simulating external services) ---

def mock_web_search(query: str) -> str:
    """Simulates a web search and returns relevant information."""
    print(f"\n[TOOL] Performing web search for: '{query}'...")
    # In a real scenario, this would call a search API like Google Search, Brave Search, etc.
    # For this exercise, we return a canned response.
    if "AI agent safety" in query:
        return "Recent advancements in AI agent safety focus on robust alignment, interpretability, and control mechanisms. Key areas include formal verification of agent behavior, adversarial training for robustness, and human-in-the-loop oversight. Organizations like DeepMind, OpenAI, and Anthropic are leading research in this domain. Emerging standards for responsible AI development are also gaining traction."
    elif "LangGraph applications" in query:
        return "LangGraph is increasingly used for building complex, stateful AI agents. Its applications span multi-agent systems, conversational AI, autonomous task execution, and dynamic workflow orchestration. Examples include customer support bots with memory, research assistants, and automated code generation pipelines. The ability to define cyclic graphs and manage state makes it ideal for iterative processes."
    else:
        return f"Simulated search results for '{query}': Found general information about the topic, including historical context and current trends. Specific details might require more focused queries."

def mock_data_analysis(data: str) -> str:
    """Simulates data analysis on provided research notes."""
    print(f"\n[TOOL] Analyzing research notes (first 50 chars): '{data[:50]}'...")
    # In a real scenario, this would involve NLP models, statistical analysis, etc.
    # For this exercise, we provide a summary based on keywords.
    if "AI agent safety" in data:
        return "The analysis highlights the critical need for robust safety protocols in advanced AI agents. Key themes are alignment with human values, transparent decision-making, and mechanisms for human intervention. The current research landscape shows a strong emphasis on proactive safety measures rather than reactive ones. Gaps exist in real-world deployment and continuous monitoring."
    elif "LangGraph applications" in data:
        return "Analysis reveals LangGraph's strength in orchestrating complex AI workflows. Its stateful nature is a significant advantage for multi-turn interactions and iterative tasks. The primary applications are in automating knowledge work and enhancing interactive AI. Future potential lies in integrating with real-time data streams and more sophisticated reasoning modules."
    else:
        return f"Analysis of the provided data: Identified several key points and potential areas for further investigation. The data suggests a general understanding of the topic, but specific actionable insights are limited without more focused information. Consider refining the research query."

print("Setup complete. Mock LLM and tools are ready.")


### Student Implementation Task

Now it's your turn! Implement the following components based on the requirements:

1.  **Define Agent Nodes**: Create Python functions for each of the specialized agents (`researcher_node`, `analyzer_node`, `report_writer_node`) and the `supervisor_node`. Each function should take the `AgentState` as input, perform its specific task, and return an updated `AgentState`.
    *   Remember that the `Supervisor` agent's primary role is to decide the `next_agent` based on the current state.
    *   The other agents should update their respective fields in the state and, if they don't explicitly route, they should return control to the `Supervisor` by setting `next_agent` to `'supervisor'` (unless they are the final step).

2.  **Construct the LangGraph**: Instantiate a `StateGraph` with your `AgentState`.
    *   Add each agent function as a node to the graph.
    *   Define the entry point for the graph (it should be the `supervisor`).
    *   Set up conditional edges from the `supervisor` to route to the appropriate sub-agent based on its decision.
    *   Set up normal edges from each sub-agent back to the `supervisor` (for iterative processing) or to `END` (for final completion).

3.  **Compile and Run**: Compile your graph into an executable `app` and test it with a research query. Print the final report.

Feel free to refer to the `AgentState` and mock tools provided in the setup cell. Good luck with your implementation!


In [ ]:
### Reference Solution

# --- Agent Definitions ---

# Helper function to create an LLM chain for agents
def create_agent_chain(system_prompt: str):
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder("messages", optional=True)
    ])
    return prompt | llm # Using the mock LLM defined in setup

# 1. Researcher Agent
def researcher_node(state: AgentState) -> AgentState:
    """Gathers information based on the query using mock web search."""
    print("\n--- RESEARCHER AGENT ---")
    query = state["query"]
    current_notes = state.get("research_notes", "")

    # Define the researcher's prompt
    researcher_prompt = create_agent_chain(
        f"You are a diligent researcher. Your task is to gather information for the given query using web search. "
        f"Current query: {query}\n"
        f"Existing research notes: {current_notes}\n"
        f"Based on the query, perform a web search and summarize key findings. Focus on adding new, relevant information. "
        f"If you believe enough research has been done, indicate that. Otherwise, provide new findings."
    )

    # Simulate web search and LLM processing
    search_results = mock_web_search(query) # Use the mock tool
    
    # In a real scenario, the LLM would process search_results to refine notes.
    # For this exercise, we'll directly append and let the supervisor decide if more is needed.
    new_notes = f"\n- Search for '{query}': {search_results}"
    updated_notes = current_notes + new_notes

    # Update state and route back to supervisor for next decision
    return {
        "research_notes": updated_notes,
        "next_agent": "supervisor", # Always route back to supervisor for decision
        "messages": state.get("messages", []) + [AIMessage(content=f"Researcher found new notes. Current notes length: {len(updated_notes)}")]
    }

# 2. Analyzer Agent
def analyzer_node(state: AgentState) -> AgentState:
    """Analyzes research notes and synthesizes insights."""
    print("\n--- ANALYZER AGENT ---")
    research_notes = state["research_notes"]
    current_analysis = state.get("analysis_results", "")

    # Define the analyzer's prompt
    analyzer_prompt = create_agent_chain(
        f"You are a critical analyzer. Your task is to synthesize the provided research notes, identify key insights, "
        f"and point out any gaps or areas requiring further investigation. "
        f"Research Notes: {research_notes}\n"
        f"Existing Analysis: {current_analysis}\n"
        f"Provide a concise summary of the main findings and suggest if more research is needed on specific points. "
        f"Format your output as a clear analysis."
    )

    # Simulate data analysis and LLM processing
    analysis_output = mock_data_analysis(research_notes) # Use the mock tool
    
    # In a real scenario, the LLM would refine the analysis.
    updated_analysis = current_analysis + f"\n- Analysis: {analysis_output}"

    # Update state and route back to supervisor for next decision
    return {
        "analysis_results": updated_analysis,
        "next_agent": "supervisor", # Always route back to supervisor for decision
        "messages": state.get("messages", []) + [AIMessage(content=f"Analyzer completed analysis. Current analysis length: {len(updated_analysis)}")]
    }

# 3. Report Writer Agent
def report_writer_node(state: AgentState) -> AgentState:
    """Generates the final research report based on analysis results."""
    print("\n--- REPORT WRITER AGENT ---")
    analysis_results = state["analysis_results"]
    query = state["query"]

    # Define the report writer's prompt
    report_writer_prompt = create_agent_chain(
        f"You are a professional report writer. Your task is to generate a comprehensive and well-structured research report. "
        f"Initial Query: {query}\n"
        f"Analysis Results: {analysis_results}\n"
        f"Based on the analysis, write a detailed report including an executive summary, key findings, and conclusions. "
        f"Ensure the report is coherent and addresses the initial query fully."
    )

    # Simulate report generation
    # For this exercise, we'll use a simple LLM call to format the report.
    report_content = llm.invoke(report_writer_prompt.invoke({"messages": [HumanMessage(content=f"Generate report for query: {query}")]})).content

    # Update state and indicate completion
    return {
        "final_report": report_content,
        "next_agent": "FINISH", # This agent signals the end of the process
        "messages": state.get("messages", []) + [AIMessage(content="Report Writer completed the final report.")]
    }

# 4. Supervisor Agent
def supervisor_node(state: AgentState) -> AgentState:
    """Decides the next agent to route to based on the current state and task progress."""
    print("\n--- SUPERVISOR AGENT ---")
    query = state["query"]
    research_notes = state.get("research_notes", "")
    analysis_results = state.get("analysis_results", "")
    final_report = state.get("final_report", "")
    iterations = state.get("iterations", 0) + 1

    # Prevent infinite loops
    if iterations > 5: # Set a reasonable iteration limit for the exercise
        print("\n[SUPERVISOR] Max iterations reached. Forcing FINISH.")
        return {"next_agent": "FINISH", "iterations": iterations, "messages": state.get("messages", []) + [AIMessage(content="Supervisor: Max iterations reached.")]}

    # Define the supervisor's prompt
    supervisor_prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a research supervisor. Your role is to direct the workflow of a multi-agent research team. "
         "Based on the current state of the research, decide which agent should act next. "
         "Available agents: `researcher`, `analyzer`, `report_writer`. "
         "You can also decide to `FINISH` if the research query has been fully addressed and a report is ready. "
         "Your output MUST be a JSON object with a single key `next_agent` and its value being one of the allowed agents or `FINISH`."
         f"\n\nCurrent Query: {query}"
         f"\nResearch Notes (length: {len(research_notes)}): {research_notes[:200]}..."
         f"\nAnalysis Results (length: {len(analysis_results)}): {analysis_results[:200]}..."
         f"\nFinal Report (length: {len(final_report)}): {final_report[:200]}..."
         f"\nIteration: {iterations}"
         f"\n\nBased on this, decide the next agent to route to."
        ),
        MessagesPlaceholder("messages", optional=True)
    ])

    # Use LLM to decide the next agent
    # The mock LLM has simplified logic, a real LLM would use the full context.
    response = llm.invoke(supervisor_prompt.invoke({"messages": state.get("messages", [])})).content
    
    try:
        decision = json.loads(response)
        next_agent = decision.get("next_agent", "FINISH") # Default to FINISH if parsing fails
    except json.JSONDecodeError:
        print(f"[SUPERVISOR ERROR] Could not parse LLM response: {response}. Defaulting to FINISH.")
        next_agent = "FINISH"

    print(f"[SUPERVISOR] Decided next agent: {next_agent}")
    return {
        "next_agent": next_agent,
        "iterations": iterations,
        "messages": state.get("messages", []) + [AIMessage(content=f"Supervisor routed to: {next_agent}")]
    }

# --- Graph Construction ---

# Initialize the StateGraph with our AgentState
workflow = StateGraph(AgentState)

# Add nodes for each agent
workflow.add_node("researcher", researcher_node)
workflow.add_node("analyzer", analyzer_node)
workflow.add_node("report_writer", report_writer_node)
workflow.add_node("supervisor", supervisor_node)

# Set the entry point for the graph
workflow.set_entry_point("supervisor")

# Define conditional edges from the supervisor
# The supervisor's output 'next_agent' determines the next step.
workflow.add_conditional_edges(
    "supervisor", # From supervisor
    lambda state: state["next_agent"], # Based on the 'next_agent' key in state
    {
        "researcher": "researcher",
        "analyzer": "analyzer",
        "report_writer": "report_writer",
        "FINISH": END # If supervisor decides to FINISH, terminate the graph
    }
)

# Define edges from sub-agents back to the supervisor
# After a sub-agent completes its task, it returns control to the supervisor
# for the next decision, unless it's the final step (report_writer).
workflow.add_edge("researcher", "supervisor")
workflow.add_edge("analyzer", "supervisor")

# The report_writer directly leads to END, as it's the final step.
workflow.add_edge("report_writer", END)

# Compile the graph
app = workflow.compile()

print("\nLangGraph compiled successfully. Ready to run the research system.\n")

# --- Run the Research System ---

# Example Research Query 1
print("\n====================================================")
print("RUNNING RESEARCH SYSTEM FOR: AI agent safety best practices")
print("====================================================")

initial_state_1 = {
    "query": "AI agent safety best practices",
    "research_notes": "",
    "analysis_results": "",
    "final_report": "",
    "next_agent": "",
    "iterations": 0,
    "messages": [HumanMessage(content="Start research on AI agent safety best practices.")]
}

final_state_1 = app.invoke(initial_state_1)

print("\n--- FINAL REPORT (Query 1) ---")
print(final_state_1["final_report"])
print("\nFinal State Iterations:", final_state_1["iterations"])

# Example Research Query 2
print("\n====================================================")
print("RUNNING RESEARCH SYSTEM FOR: Advanced LangGraph applications in 2026")
print("====================================================")

initial_state_2 = {
    "query": "Advanced LangGraph applications in 2026",
    "research_notes": "",
    "analysis_results": "",
    "final_report": "",
    "next_agent": "",
    "iterations": 0,
    "messages": [HumanMessage(content="Start research on Advanced LangGraph applications in 2026.")]
}

final_state_2 = app.invoke(initial_state_2)

print("\n--- FINAL REPORT (Query 2) ---")
print(final_state_2["final_report"])
print("\nFinal State Iterations:", final_state_2["iterations"])
